In [ ]:
# fix initial position for pipette


# Init

In [1]:
from workspace import Workspace
from util import create_recipes

# workspace
workspace = Workspace(config_path=["config/base.j2", "config/layout.j2"])
core = workspace.components["core"]

# recepies
rcp = create_recipes(workspace, core)

✅ core connected @ 192.168.254.124
🔵 🕹️ simulation api enabled
✅ printer connected @ 192.168.254.128
✅ pipettor connected @ /dev/ttyUSB0
[Display] socket.io connected
[Display] sending initial snapshot (145 items)
[Display] Running at 60 fps


# parameters

In [2]:
# tip list
tip_list = [f"{r}{c}" for r in "H" for c in range(12, 13)]

# tube
tube_list = [f"{r}{c}" for r in "AB" for c in range(1,2)]
falcon_rack_gravity_offset = 2

# cap
cap_list = [f"{r}{c}" for r in "CD" for c in range(1,2)]
cap_offset = [0, 0, 111, 0, 0, 0]

# pipette
vol = 400 #ul
immerse_depth = 20
#tool_rack_1_joint = [-34.628906, 43.527832, -80.15625, 1.07666, -53.10791, -32.827148, 220.575, 0]
tool_rack_1_joint = [-34.628906, 43.527832, -80.15625, 1.07666, -53.10791, -32.827148, 263.0625, 0]

# printer
dry_run_count = 1

# inspection
inspection_frq = 4
inspection_rot = 90

# tool_rack_0
tool_rack_0_joint = [-14.39209, 40.297852, -94.614258, -0.219727, -35.024414, -13.688965, 121.89375, 0]

# simulation

In [3]:
core.simulation(False)
workspace.components["pipettor"].simulation(False)
workspace.components["printer"].simulation(False)

🟡 🕹️ simulation api disabled


# main loop

In [ ]:
for tip_index in tip_list:
    # pick pipettor
    rcp["tool_rack_1"].pick()
    core.robot_api.jmove(joint=tool_rack_1_joint, 
                        vel=rcp["tool_rack_1"].jmove_vaj[0]*rcp["tool_rack_1"].speed_factor, 
                        accel=rcp["tool_rack_1"].jmove_vaj[1]*rcp["tool_rack_1"].speed_factor, 
                        jerk=rcp["tool_rack_1"].jmove_vaj[2]*rcp["tool_rack_1"].speed_factor)

    # pick tip
    rcp["tip_rack"].pick_tip(tip_index)

    # pipette all the tubes
    for i in range(len(tube_list)):
        # source and destination
        aspirate_index = tube_list[i]
        dispense_index = tube_list[len(tube_list)-(i+1)]

        # immerse source and asspirate and retract
        rcp["falcon_pipepette"].immerse(anchor=aspirate_index, depth=immerse_depth)
        rcp["falcon_pipepette"].aspirate(vol=vol)
        rcp["falcon_pipepette"].retract(anchor=aspirate_index)

        # immerse target and dispense and retract
        rcp["falcon_pipepette"].immerse(anchor=dispense_index, depth=immerse_depth)
        rcp["falcon_pipepette"].dispense(vol=vol)
        rcp["falcon_pipepette"].retract(anchor=dispense_index)

    # eject tip
    rcp["waste_bin"].eject_tip(shake_travel=8)

    # place pipettor
    rcp["tool_rack_1"].place()

    # pick gripper
    rcp["tool_rack_0"].pick()
    core.robot_api.jmove(joint=tool_rack_0_joint, 
                        vel=rcp["tool_rack_0"].jmove_vaj[0]*rcp["tool_rack_0"].speed_factor, 
                        accel=rcp["tool_rack_0"].jmove_vaj[1]*rcp["tool_rack_0"].speed_factor, 
                        jerk=rcp["tool_rack_0"].jmove_vaj[2]*rcp["tool_rack_0"].speed_factor)
 

    # loop over all tubes, cap, print and inspect
    for tube_index, cap_index in zip(tube_list, cap_list):
        # pick tube
        rcp["falcon_rack"].pick_from(tube_index)

        # put in decapper
        rcp["decapper"].place()

        # pick cap
        rcp["falcon_rack"].pick_from(cap_index)

        # capping and pick tube
        rcp["decapper"].cap(exit=False)
        rcp["decapper"].pick(approach=False)

        # print
        rcp["printer"].place(exit=False)
        rcp["printer"].dry_run_spin(count=dry_run_count)
        rcp["printer"].pick(approach=False)

        # inspection present and rotate
        rcp["inspector"].present(approach=False)
        for i in range(inspection_frq):
            rcp["inspector"].rotate(rotation=inspection_rot)

        # place tube
        rcp["falcon_rack"].place_in(tube_index, gravity_offset=falcon_rack_gravity_offset)

    # decap all the tubes 
    for tube_index, cap_index in zip(tube_list, cap_list):
        # pick tube
        rcp["falcon_rack"].pick_from(tube_index)  

        # put in decapper, decap
        rcp["decapper"].place(exit=False)
        rcp["decapper"].decap(approach=False)

        # place cap
        rcp["falcon_rack"].place_in(cap_index, offset=cap_offset)

        # pick tube
        rcp["decapper"].pick()

        # place tube
        rcp["falcon_rack"].place_in(tube_index, gravity_offset=falcon_rack_gravity_offset)
    
    # place gripper
    core.robot_api.jmove(joint=tool_rack_0_joint, 
                        vel=rcp["tool_rack_0"].jmove_vaj[0]*rcp["tool_rack_0"].speed_factor, 
                        accel=rcp["tool_rack_0"].jmove_vaj[1]*rcp["tool_rack_0"].speed_factor, 
                        jerk=rcp["tool_rack_0"].jmove_vaj[2]*rcp["tool_rack_0"].speed_factor)
    rcp["tool_rack_0"].place()